# 단계 15. 인제군 전이 시험 (재학습 없이 춘천 모델 적용)

논문 3.8·4.8절, Table 11 · 절차서: `forest-change-agent/EXPERIMENTS.md` 순서 15

입력: GEE 자산 `projects/ee-choish7838/assets/inje_transfer`(`scripts/11_inje_prepare.py` 산출을 업로드)

산출: `outputs/results/E10_inje_transfer.json`, `work/inje/visual_check_points_raw.csv` → `scripts/13_inje_visual_check.py`

같은 내용의 스크립트판: `scripts/12_inje_transfer.py`

> 셀 출력은 원 실행 기록입니다(원본 노트북: scratchpad (7).ipynb). 당시 이름(Mini-GFM, `mini_gfm_ts.py`, `GEE_Export/mini_gfm_chuncheon` 등)은 코드에서 S2-TSFM·단계 번호 이름으로 바꿨고, 출력 문구는 실행 당시 그대로입니다.


In [1]:
# [15-1] 인제군 전이 시험 — 1. GEE 연결
import ee
PROJECT = 'ee-choish7838'
try:
    ee.Initialize(project=PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROJECT)
print('GEE ok', ee.Number(1).getInfo())

GEE ok 1


In [2]:
# [15-2] 특징 정의 — forest-change-agent/fcd/gee.py·features.py 와 동일
import base64, gzip, io, time
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd

AE_YEARS = list(range(2017, 2026)); AE_BANDS = [f'A{i:02d}' for i in range(64)]
Y0, Y1, NODATA, SEED = 2019, 2025, -999.0, 42
CLASSES = ['no_change', 'harvest', 'conversion', 'gain']

def ae_year(y):
    return ee.ImageCollection('GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL').filter(ee.Filter.calendarRange(y, y, 'year')).mosaic().select(AE_BANDS)

def s2_summer(y):
    def mask(img):
        scl = img.select('SCL')
        return img.updateMask(scl.neq(3).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10)).And(scl.neq(11)))
    med = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED').filterDate(f'{y}-06-01', f'{y}-09-30')
           .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 70)).map(mask).median())
    return med.normalizedDifference(['B8', 'B4']).rename(f'ndvi{y}').addBands(med.normalizedDifference(['B8', 'B12']).rename(f'nbr{y}'))

def feature_image():
    img = ee.Image.cat([ae_year(y).rename([f'ae{y}_{b}' for b in AE_BANDS]) for y in AE_YEARS])
    for y in (Y0, Y1):
        img = img.addBands(s2_summer(y))
    dem = ee.Image('USGS/SRTMGL1_003').select('elevation')
    return img.addBands(dem.rename('elev')).addBands(ee.Terrain.slope(dem).rename('slope')).unmask(NODATA)

IMG = feature_image()

def _chunk(c):
    feats = [ee.Feature(ee.Geometry.Point([r.lon, r.lat]), {'sample_id': r.sample_id}) for r in c.itertuples()]
    for k in range(4):
        try:
            fc = IMG.sampleRegions(collection=ee.FeatureCollection(feats), properties=['sample_id'], scale=10, tileScale=4, geometries=False)
            return [f['properties'] for f in fc.getInfo()['features']]
        except Exception as e:
            time.sleep(5 * (k + 1)); err = e
    raise err

def extract(points, chunk=150, workers=6):
    parts = [points.iloc[i:i + chunk] for i in range(0, len(points), chunk)]
    rows, t = [], time.time()
    with ThreadPoolExecutor(workers) as ex:
        for i, res in enumerate(ex.map(_chunk, parts), 1):
            rows += res
            if i % 5 == 0 or i == len(parts):
                print(f'  {i}/{len(parts)} 청크, {len(rows):,}점, {time.time() - t:.0f}s', flush=True)
    df = pd.DataFrame(rows); num = df.columns.drop('sample_id')
    df[num] = df[num].astype('float32').replace(NODATA, np.nan)
    return df

def _ae(df, y):
    e = df[[f'ae{y}_{b}' for b in AE_BANDS]].to_numpy('float32')
    return e / np.clip(np.linalg.norm(e, axis=1, keepdims=True), 1e-6, None)

def build(df):
    F = {}; e0, e1 = _ae(df, Y0), _ae(df, Y1)
    for i, b in enumerate(AE_BANDS):
        F[f'ae_diff_{b}'] = e1[:, i] - e0[:, i]
    F['ae_cos_dist'] = 1 - (e0 * e1).sum(1)
    embs = {y: _ae(df, y) for y in AE_YEARS}
    consec = {y: 1 - (embs[y - 1] * embs[y]).sum(1) for y in AE_YEARS[1:]}
    for y, d in consec.items():
        F[f'ae_d{y}'] = d
    win = [y for y in consec if Y0 < y <= Y1]; pre = [y for y in consec if y <= Y0]
    W = np.stack([consec[y] for y in win], 1)
    F['ae_consec_max'] = W.max(1); F['ae_peak_year'] = np.array(win)[W.argmax(1)]
    F['ae_pre_noise'] = np.stack([consec[y] for y in pre], 1).max(1)
    for idx in ('ndvi', 'nbr'):
        F[f'{idx}_{Y0}'] = df[f'{idx}{Y0}'].to_numpy(); F[f'{idx}_{Y1}'] = df[f'{idx}{Y1}'].to_numpy()
        F[f'd{idx}'] = F[f'{idx}_{Y1}'] - F[f'{idx}_{Y0}']
    F['elev'], F['slope'] = df['elev'].to_numpy(), df['slope'].to_numpy()
    X = pd.DataFrame(F, index=df.index)
    G = {'AE': [c for c in X if c.startswith('ae_') and c != 'ae_peak_year'],
         'S2': [c for c in X if c.startswith(('ndvi', 'nbr', 'dndvi', 'dnbr'))], 'TERR': ['elev', 'slope']}
    return X, G

def load_payload(txt):
    return pd.read_csv(io.BytesIO(gzip.decompress(base64.b64decode(txt))))
print('특징 정의 완료:', len(AE_YEARS), '개 연도')

특징 정의 완료: 9 개 연도


In [3]:
# [15-3] 표본점 불러오기(GEE 자산) → 두 지역 특징 추출
ROOT = 'projects/ee-choish7838/assets/inje_transfer'
def read_asset(name):
    for _ in range(60):
        try:
            fc = ee.FeatureCollection(f'{ROOT}/{name}'); n = fc.size().getInfo(); break
        except Exception:
            time.sleep(10)
    rows = []
    for off in range(0, n, 5000):
        for f in fc.toList(5000, off).getInfo():
            p = f['properties']; p['lon'], p['lat'] = f['geometry']['coordinates']; rows.append(p)
    return pd.DataFrame(rows)

pts = {k: read_asset(k) for k in ('chuncheon_samples', 'inje_samples', 'inje_events')}
for k, v in pts.items():
    print(k, v.shape)
FEAT = {}
for k in ('chuncheon_samples', 'inje_samples', 'inje_events'):
    print('추출:', k)
    FEAT[k] = pts[k].merge(extract(pts[k]), on='sample_id')
    FEAT[k].to_parquet(f'/content/{k}_features.parquet')
    print(k, FEAT[k].shape, '결측 AE 행', int(FEAT[k]['ae2025_A00'].isna().sum()))

chuncheon_samples (6027, 7)
inje_samples (2100, 7)
inje_events (156, 6)
추출: chuncheon_samples
  5/41 청크, 750점, 5s
  10/41 청크, 1,500점, 10s
  15/41 청크, 2,250점, 14s
  20/41 청크, 3,000점, 15s
  25/41 청크, 3,750점, 22s
  30/41 청크, 4,500점, 22s
  35/41 청크, 5,250점, 25s
  40/41 청크, 6,000점, 28s
  41/41 청크, 6,027점, 28s
chuncheon_samples (6027, 589) 결측 AE 행 0
추출: inje_samples
  5/14 청크, 750점, 9s
  10/14 청크, 1,500점, 13s
  14/14 청크, 2,100점, 14s
inje_samples (2100, 589) 결측 AE 행 0
추출: inje_events
  2/2 청크, 156점, 4s
inje_events (156, 588) 결측 AE 행 0


In [4]:
# [15-4] 평가: (a) 춘천 재현 (b) 춘천 모델 → 인제 전이(재학습 없음) (c) 인제 자체 공간블록 CV (d) 인제 사업실적 시점
import json
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix
from sklearn.model_selection import StratifiedGroupKFold

def model():
    return HistGradientBoostingClassifier(max_iter=300, learning_rate=0.06, max_leaf_nodes=31, l2_regularization=1.0,
                                          class_weight='balanced', random_state=SEED)

ch, ij = FEAT['chuncheon_samples'], FEAT['inje_samples']
Xc, G = build(ch); Xi, _ = build(ij)
yc, yi = (ch['cls'] > 0).astype(int), (ij['cls'] > 0).astype(int)
tr, te = ch['split'].eq('train'), ch['split'].eq('test_spatial')
COMBOS = {'AE': G['AE'], 'S2': G['S2'], 'TERR': G['TERR'], 'AE+S2': G['AE'] + G['S2'], 'AE+S2+TERR': G['AE'] + G['S2'] + G['TERR']}

def cv_f1(X, y, g, average='binary'):
    s = []
    for a, b in StratifiedGroupKFold(5, shuffle=True, random_state=SEED).split(X, y, g):
        s.append(f1_score(y.iloc[b], model().fit(X.iloc[a], y.iloc[a]).predict(X.iloc[b]), average=average))
    return np.mean(s), np.std(s)

R = []
for name, cols in COMBOS.items():
    m = model().fit(Xc.loc[tr, cols], yc[tr])
    f_ch = f1_score(yc[te], m.predict(Xc.loc[te, cols]))
    p = m.predict(Xi[cols])
    cvm, cvs = cv_f1(Xi[cols], yi, ij['block_id'])
    R.append({'features': name, 'n_feat': len(cols), 'chuncheon_holdout_F1': round(f_ch, 3),
              'inje_transfer_F1': round(f1_score(yi, p), 3), 'inje_transfer_P': round(precision_score(yi, p), 3),
              'inje_transfer_R': round(recall_score(yi, p), 3), 'inje_local_CV_F1': f'{cvm:.3f} ± {cvs:.3f}'})
T1 = pd.DataFrame(R); print(T1.to_string(index=False))

# 전이 모델(AE+S2+TERR) 세부: 혼동행렬, 유형별 미탐지
cols = COMBOS['AE+S2+TERR']
m = model().fit(Xc.loc[tr, cols], yc[tr]); p = m.predict(Xi[cols])
cm = confusion_matrix(yi, p).tolist()
fn = pd.DataFrame({'type': ij['cls'].map(dict(enumerate(CLASSES))), 'miss': (yi == 1) & (p == 0)})
fn = fn[fn['type'] != 'no_change'].groupby('type')['miss'].agg(['size', 'sum']); fn['rate'] = (fn['sum'] / fn['size']).round(3)
print('\n혼동행렬 [[TN,FP],[FN,TP]]', cm); print(fn.to_string())

# T2 변화유형: 춘천 학습 → 인제, 인제 자체 CV
TYPES = ['harvest', 'conversion', 'gain']
cc, ci = ch['cls'] > 0, ij['cls'] > 0
yt_c, yt_i = ch['cls'].map(dict(enumerate(CLASSES))), ij['cls'].map(dict(enumerate(CLASSES)))
mt = model().fit(Xc.loc[tr & cc, cols], yt_c[tr & cc])
pt_i = mt.predict(Xi.loc[ci, cols]); pt_c = mt.predict(Xc.loc[te & cc, cols])
t2 = {'chuncheon_holdout_macroF1': round(f1_score(yt_c[te & cc], pt_c, average='macro'), 3),
      'inje_transfer_macroF1': round(f1_score(yt_i[ci], pt_i, average='macro'), 3),
      'inje_transfer_classF1': dict(zip(TYPES, f1_score(yt_i[ci], pt_i, labels=TYPES, average=None).round(3).tolist())),
      'inje_local_CV_macroF1': '%.3f ± %.3f' % cv_f1(Xi.loc[ci, cols], yt_i[ci], ij.loc[ci, 'block_id'], 'macro'),
      'inje_transfer_confusion(rows=true)': confusion_matrix(yt_i[ci], pt_i, labels=TYPES).tolist()}
print('\nT2', json.dumps(t2, ensure_ascii=False))

# (d) 인제 사업실적(2020~2024 주벌·조림예정지정리·인공갱신) vs AE 연속연도 거리 정점
ev = FEAT['inje_events']
emb = {y: _ae(ev, y) for y in AE_YEARS}
D = pd.DataFrame({y: 1 - (emb[y - 1] * emb[y]).sum(1) for y in range(2018, 2026)}); D['unit'] = ev['unit'].values
u = D.groupby('unit').mean(); peak = u.idxmax(axis=1).astype(int)
info = ev.groupby('unit')[['year', 'grade']].first().join(peak.rename('ae_peak'))
info['lag'] = info['ae_peak'] - info['year']
def agree(d):
    return {'n': len(d), 'exact': round(float((d.lag == 0).mean()), 3), 'within1': round(float((d.lag.abs() <= 1).mean()), 3),
            'lag_dist': {int(k): int(v) for k, v in d.lag.value_counts().sort_index().items()}}
t5 = {'all': agree(info), 'A_final_felling': agree(info[info.grade == 'A']), 'B_regeneration': agree(info[info.grade == 'B'])}
print('\n사업실적 시점', json.dumps(t5, ensure_ascii=False))
json.dump({'T1': R, 'T1_transfer_confusion': cm, 'T1_transfer_FN_by_type': fn.reset_index().to_dict('records'), 'T2': t2, 'events': t5},
          open('/content/inje_transfer_results.json', 'w'), ensure_ascii=False, indent=1)

  features  n_feat  chuncheon_holdout_F1  inje_transfer_F1  inje_transfer_P  inje_transfer_R inje_local_CV_F1
        AE      75                 0.933             0.778            0.955            0.656    0.792 ± 0.119
        S2       6                 0.764             0.755            0.828            0.694    0.727 ± 0.062
      TERR       2                 0.487             0.378            0.572            0.282    0.474 ± 0.059
     AE+S2      81                 0.936             0.823            0.960            0.721    0.847 ± 0.082
AE+S2+TERR      83                 0.934             0.760            0.969            0.625    0.843 ± 0.089

혼동행렬 [[TN,FP],[FN,TP]] [[978, 22], [412, 688]]
            size  sum   rate
type                        
conversion   300  195  0.650
gain         300  194  0.647
harvest      500   23  0.046

T2 {"chuncheon_holdout_macroF1": 0.785, "inje_transfer_macroF1": 0.776, "inje_transfer_classF1": {"harvest": 0.856, "conversion": 0.804, "gain": 0

In [5]:
# [15-5] 진단: 클래스별 AE 거리 중앙값(두 지역), 확률 임계값별 전이 F1·재현율
med = pd.DataFrame({'chuncheon': Xc.groupby(ch['cls'].map(dict(enumerate(CLASSES))))['ae_cos_dist'].median(),
                    'inje': Xi.groupby(ij['cls'].map(dict(enumerate(CLASSES))))['ae_cos_dist'].median()}).round(3)
print(med.to_string())
for name in ('AE', 'AE+S2', 'AE+S2+TERR'):
    cols = COMBOS[name]; pr = model().fit(Xc.loc[tr, cols], yc[tr]).predict_proba(Xi[cols])[:, 1]
    out = []
    for th in (0.5, 0.3, 0.2, 0.1):
        p = (pr >= th).astype(int)
        rec = {t: round(float(p[(ij['cls'] == k).values].mean()), 3) for k, t in enumerate(CLASSES) if k > 0}
        out.append(f"th {th}: F1 {f1_score(yi, p):.3f} P {precision_score(yi, p):.3f} R {recall_score(yi, p):.3f} {rec}")
    print(name, *out, sep='\n  ')

            chuncheon   inje
cls                         
conversion      0.042  0.039
gain            0.080  0.083
harvest         0.211  0.206
no_change       0.031  0.025
AE
  th 0.5: F1 0.778 P 0.955 R 0.656 {'harvest': 0.96, 'conversion': 0.317, 'gain': 0.49}
  th 0.3: F1 0.808 P 0.938 R 0.710 {'harvest': 0.964, 'conversion': 0.41, 'gain': 0.587}
  th 0.2: F1 0.825 P 0.924 R 0.745 {'harvest': 0.968, 'conversion': 0.467, 'gain': 0.653}
  th 0.1: F1 0.844 P 0.896 R 0.798 {'harvest': 0.976, 'conversion': 0.557, 'gain': 0.743}
AE+S2
  th 0.5: F1 0.823 P 0.960 R 0.721 {'harvest': 0.966, 'conversion': 0.577, 'gain': 0.457}
  th 0.3: F1 0.849 P 0.949 R 0.767 {'harvest': 0.972, 'conversion': 0.65, 'gain': 0.543}
  th 0.2: F1 0.861 P 0.932 R 0.800 {'harvest': 0.974, 'conversion': 0.677, 'gain': 0.633}
  th 0.1: F1 0.866 P 0.912 R 0.825 {'harvest': 0.974, 'conversion': 0.713, 'gain': 0.69}
AE+S2+TERR
  th 0.5: F1 0.760 P 0.969 R 0.625 {'harvest': 0.954, 'conversion': 0.35, 'gain': 0.353}
  